# task_04 Solution

In [1]:

from pathlib import Path
import pandas as pd

base = Path("../../tasks/task_04/data/")


In [2]:
batches = pd.read_csv(base / "batches.csv")
maintenance = pd.read_csv(base / "maintenance.csv")
qc = pd.read_csv(base / "qc.csv")
machines = pd.read_csv(base / "machines.csv")

Q1: For post-maintenance batches (strictly within 7 days after any maintenance), compute energy per good unit. Which line has the worst (highest) median?

In [3]:
batches["batch_date"] = pd.to_datetime(batches["batch_date"])
maintenance["maintenance_date"] = pd.to_datetime(maintenance["maintenance_date"])

merged = batches.merge(qc, on="batch_id").merge(machines, on="machine_id")

# Identify post-maintenance batches: strictly within 7 days after any maintenance event
def is_post_maint(row):
    machine_maint = maintenance[maintenance["machine_id"] == row["machine_id"]]
    for _, m in machine_maint.iterrows():
        days = (row["batch_date"] - m["maintenance_date"]).days
        if 0 < days <= 7:
            return True
    return False

merged["post_maint"] = merged.apply(is_post_maint, axis=1)
post = merged[merged["post_maint"]].copy()
post["good_units"] = post["units_produced"] - post["defect_units"] - post["rework_units"]
post["epgu"] = post["energy_kwh"] / post["good_units"]

q1 = post.groupby("line")["epgu"].median().idxmax()
print(q1)

A


Q2: Per-machine rework-to-defect ratio. Among machines with avg runtime strictly above overall avg, which has the highest ratio?

In [4]:
all_merged = batches.merge(qc, on="batch_id")
overall_avg_runtime = all_merged.merge(machines, on="machine_id")["runtime_hours"].mean() if "runtime_hours" in batches.columns else batches["runtime_hours"].mean()

# Actually merge properly
bm = batches.merge(qc, on="batch_id")
overall_avg = bm["runtime_hours"].mean()
machine_avg = bm.groupby("machine_id")["runtime_hours"].mean()
eligible = machine_avg[machine_avg > overall_avg].index

machine_totals = bm.groupby("machine_id").agg(
    total_rework=("rework_units", "sum"),
    total_defects=("defect_units", "sum")
)
machine_totals["ratio"] = machine_totals["total_rework"] / machine_totals["total_defects"]
eligible_ratios = machine_totals.loc[eligible].sort_values(["ratio", "ratio"], ascending=[False, True])

q2 = eligible_ratios.index[0]
print(q2)

M05
